# OpenManus on Kaggle (free LLM APIs: Groq / Gemini / OpenRouter)This notebook runs the **OpenManus** agent (`himanshuverma993/OpenManus`) on a KaggleCPU notebook using a **free** LLM provider.Everything below is self-contained: it installs the (slim) dependency set, clones therepo, resolves your API key from **Kaggle Secrets**, writes `config/config.toml`, andruns the agent.---## What you need before running1. A free API key from one of:   * **Groq** - <https://console.groq.com/keys> (free tier, fastest)   * **Google Gemini** - <https://aistudio.google.com/apikey> (free tier, 1M context)   * **OpenRouter** - <https://openrouter.ai/keys> (`:free` models, but only ~50 req/day)2. In Kaggle: **Add-ons -> Secrets -> New Secret** and add it with the exact name   `GROQ_API_KEY` / `GEMINI_API_KEY` / `OPENROUTER_API_KEY`   (enable *Internet* in the notebook settings - Kaggle needs it for pip and the API).## Provider cheat sheet| Provider | `base_url` | Free model to start with | Free-tier limits ||---|---|---|---|| Groq | `https://api.groq.com/openai/v1` | `llama-3.3-70b-versatile` | 30 req/min, 1 000 req/day || Groq (cheap) | `https://api.groq.com/openai/v1` | `llama-3.1-8b-instant` | 30 req/min, 14 400 req/day || Gemini | `https://generativelanguage.googleapis.com/v1beta/openai/` | `gemini-2.5-flash` | very generous || OpenRouter | `https://openrouter.ai/api/v1` | `google/gemini-2.5-flash:free` | 20 req/min, 50 req/day |> **Tip:** Groq is the best default for an agent loop - it is fast and its free tier> allows ~1 000 requests/day, and the agent burns one request per step.

## 1 - Clone the repo and install dependencies

In [ ]:
import osimport subprocessimport sysREPO_DIR = "/kaggle/working/OpenManus"REPO_URL = "https://github.com/himanshuverma993/OpenManus.git"if not os.path.isdir(os.path.join(REPO_DIR, ".git")):    subprocess.run(        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],        check=True,    )else:    print("Repo already present ->", REPO_DIR)os.chdir(REPO_DIR)if REPO_DIR not in sys.path:    sys.path.insert(0, REPO_DIR)print("cwd:", os.getcwd())

In [ ]:
# The slim dependency set: only what the core Manus agent needs.# (skips playwright / browsergym / crawl4ai / docker / boto3 / pandas)!pip install -q --upgrade pip!pip install -q -r requirements-kaggle.txt# Pre-warm the tiktoken BPE cache so token counting is accurate.# If the download is blocked the agent still works, just with rough token counts.import tiktokentry:    tiktoken.get_encoding("cl100k_base")    print("tiktoken cache ready")except Exception as exc:    print("tiktoken cache unavailable (offline tokenizer will be used):", exc)

## 2 - Choose a provider & resolve the API keyThe cell below reads the key from **Kaggle Secrets** first and falls back to a plainenvironment variable. Pick the provider by changing `PROVIDER`.

In [ ]:
import osPROVIDER = "groq"  # one of: "groq" | "gemini" | "openrouter"PROVIDER_CONFIG = {    "groq": {        "secret": "GROQ_API_KEY",        "model": "llama-3.3-70b-versatile",        "base_url": "https://api.groq.com/openai/v1",        "max_tokens": 8192,        "temperature": 0.0,    },    "gemini": {        "secret": "GEMINI_API_KEY",        "model": "gemini-2.5-flash",        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",        "max_tokens": 8192,        "temperature": 0.0,        "supports_images": True,    },    "openrouter": {        "secret": "OPENROUTER_API_KEY",        "model": "google/gemini-2.5-flash:free",        "base_url": "https://openrouter.ai/api/v1",        "max_tokens": 4096,        "temperature": 0.0,        "supports_images": True,    },}def resolve_api_key(secret_name: str) -> str:    """Kaggle Secret -> environment variable -> empty string."""    try:        from kaggle_secrets import UserSecretClient        return UserSecretClient().get_secret(secret_name)    except Exception:        pass    for candidate in (secret_name, "OPENMANUS_API_KEY"):        value = os.environ.get(candidate, "").strip()        if value:            return value    raise RuntimeError(        f"No API key found for '{secret_name}'. Add it under "        "Add-ons -> Secrets, or export it as an environment variable."    )cfg = PROVIDER_CONFIG[PROVIDER]api_key = resolve_api_key(cfg["secret"])print(f"Provider : {PROVIDER}")print(f"Model    : {cfg['model']}")print(f"Key      : {api_key[:8]}... ({len(api_key)} chars)")

## 3 - Write `config/config.toml``config/config.toml` is git-ignored, so it does not exist after cloning - this cellcreates it. The API key is injected through an environment variable so it is neverwritten to disk.

In [ ]:
import osos.environ[cfg["secret"]] = api_key# Browser Use needs a local Chromium + `uvx`; neither exists on Kaggle, so the# MCP browser server is disabled explicitly (the agent still works fine without it).os.environ["OPENMANUS_DISABLE_BROWSER_USE"] = "1"# The API key is referenced through an environment placeholder, so the secret is# never written to disk.config_template = """[llm]model = "__MODEL__"base_url = "__BASE_URL__"api_key = "__API_KEY__"max_tokens = __MAX_TOKENS__temperature = __TEMPERATURE__"""toml = (    config_template.replace("__MODEL__", cfg["model"])    .replace("__BASE_URL__", cfg["base_url"])    .replace("__API_KEY__", "${" + cfg["secret"] + "}")    .replace("__MAX_TOKENS__", str(cfg["max_tokens"]))    .replace("__TEMPERATURE__", str(cfg["temperature"])))if cfg.get("supports_images"):    toml += "supports_images = true\n"os.makedirs("config", exist_ok=True)with open("config/config.toml", "w") as fh:    fh.write(toml)print(open("config/config.toml").read())

## 4 - Run the agent`run_agent()` is a thin wrapper around the stock `Manus` agent, so nothing about theagent's behaviour is changed - it only adds notebook-friendly printing and cleanup.

In [ ]:
import asynciofrom app.agent.manus import Manusasync def run_agent(prompt: str) -> str:    """Run the OpenManus agent and return the final result."""    agent = await Manus.create()    try:        return await agent.run(prompt)    finally:        await agent.cleanup()def ask(prompt: str) -> str:    """Synchronous helper so the agent can be called from any cell."""    return asyncio.run(run_agent(prompt))print("Ready. Call ask(...) in the next cell.")

In [ ]:
# Change the text to whatever you want the agent to do.result = ask(    "Compute the first 25 Fibonacci numbers with python, then tell me the sum.")print("\n" + "=" * 70)print("FINAL RESULT")print("=" * 70)print(result)

## Troubleshooting| Symptom | Cause / fix ||---|---|| `ValidationError: daytona_api_key` | Fixed in this branch - `DaytonaSettings` now defaults to an empty key, so a `config.toml` without a `[daytona]` section loads fine. || `StdinNotImplementedError` / notebook hangs | Fixed - `ask_human` now detects a non-interactive stdin and returns a message instead of calling `input()`. || `ModuleNotFoundError: boto3` / `docker` | Fixed - both are imported lazily now; neither is needed for a text agent. || `401 Unauthorized` | Wrong or missing key. Re-check the secret name and that you copied the key without quotes/whitespace. || `429 Too Many Requests` | Free-tier limit. Groq `llama-3.3-70b-versatile` allows 1 000 req/day; switch to `llama-3.1-8b-instant`, or lower `max_steps`. || `400 Bad Request` on `max_tokens` | The provider caps output tokens (e.g. Groq's llama-4-scout allows 8 192). Lower `max_tokens`. || `400` on `temperature` | Some reasoning models reject it - set `temperature = "none"` in `config/config.toml`. || Token counts look wrong | tiktoken could not download its BPE file; the agent falls back to an approximate tokenizer (it still runs, budgets are just rough). || Agent loops forever | Expected for small free models - set `max_steps` lower, or use a stronger model. |## Extra knobs* **More/less work per run** - `Manus.max_steps` (default 20). A free Groq model needs  several requests per task.* **Token budget** - add `max_input_tokens = 50000` under `[llm]` to cap the total input  tokens used across the whole run.* **Switch models without editing the file** - add named profiles to `config/config.toml`  (`[llm.fast]`, `[llm.vision]`, ...) and select one with  `LLM(config_name="fast")`.* **Vision** - Groq's `meta-llama/llama-4-scout-17b-16e-instruct` and every Gemini model  accept images; OpenManus detects this automatically.